# 05 - Baseline Aggregation and Statistics

This notebook recomputes the baseline summary tables from the saved CSV outputs.
It covers final return, sample efficiency, stability, probability of
improvement, robustness, and area under the learning curve.


## Load baseline data


In [1]:
from pathlib import Path
import os, sys, time

def find_baseline_root():
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        for candidate in [base, base / "baseline", base / "MuJoCo_RL_Project_Final_Submission" / "baseline"]:
            if (candidate / "baseline_summary.md").exists() and (candidate / "results").exists():
                return candidate.resolve()
    raise RuntimeError("Could not locate baseline folder")

BASE_DIR = find_baseline_root()
os.chdir(BASE_DIR)
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RAW_DIR = BASE_DIR / "results" / "raw"
PROC_DIR = BASE_DIR / "results" / "processed"
FINAL_DIR = BASE_DIR / "results" / "final"
FIG_DIR = BASE_DIR / "figures"
REPORT_FIG_DIR = FIG_DIR / "report_ready"
for folder in [RAW_DIR, PROC_DIR, FINAL_DIR, REPORT_FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ENV_IDS = ["HalfCheetah-v5", "Hopper-v5", "Walker2d-v5"]
ALGORITHMS = ["PPO", "SAC", "TD3", "DDPG", "TQC"]
print(f"Baseline root: {BASE_DIR}")
print(f"Python: {sys.executable}")


Baseline root: D:\MuJoCo_RL_Project\MuJoCo_RL_Project_Final_Submission\baseline
Python: C:\Users\digilians01\.conda\envs\RL_PROJECT\python.exe


In [2]:
eval_frames = []
for env_id in ENV_IDS:
    path = RAW_DIR / f"{env_id}_eval_log.csv"
    if path.exists():
        eval_frames.append(pd.read_csv(path))
eval_log = pd.concat(eval_frames, ignore_index=True) if eval_frames else pd.DataFrame()
final_eval = pd.read_csv(PROC_DIR / "final_eval_all.csv") if (PROC_DIR / "final_eval_all.csv").exists() else pd.DataFrame()
robust_eval = pd.read_csv(PROC_DIR / "robustness_eval_all.csv") if (PROC_DIR / "robustness_eval_all.csv").exists() else pd.DataFrame()
print(f"Eval log rows: {len(eval_log)}")
print(f"Final eval rows: {len(final_eval)}")
print(f"Robustness rows: {len(robust_eval)}")
WRITE_TABLES = False


Eval log rows: 15300
Final eval rows: 85
Robustness rows: 6800


## Statistical helpers


In [3]:
def compute_iqm(values):
    arr = np.sort(np.asarray(values, dtype=float))
    if len(arr) < 4:
        return float(np.mean(arr))
    q1 = len(arr) // 4
    q3 = len(arr) - q1
    return float(np.mean(arr[q1:q3]))

def bootstrap_ci(values, n_bootstrap=2000, rng_seed=42):
    arr = np.asarray(values, dtype=float)
    rng = np.random.RandomState(rng_seed)
    stats = [np.mean(rng.choice(arr, size=len(arr), replace=True)) for _ in range(n_bootstrap)]
    return float(np.percentile(stats, 2.5)), float(np.percentile(stats, 97.5))

def probability_of_improvement(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return float(np.mean(a[:, None] > b[None, :])) if len(a) and len(b) else np.nan


## Final return table


In [4]:
rows = []
for (env_id, algo), part in final_eval.groupby(["environment", "algorithm"]):
    returns = part["mean_return"].values
    lo, hi = bootstrap_ci(returns)
    rows.append({"environment": env_id, "algorithm": algo, "n_seeds": len(returns),
                 "mean_final_return": np.mean(returns), "std_final_return": np.std(returns),
                 "median_final_return": np.median(returns), "iqm_final_return": compute_iqm(returns),
                 "ci95_low": lo, "ci95_high": hi})
perf_df = pd.DataFrame(rows)
if WRITE_TABLES:
    perf_df.to_csv(FINAL_DIR / "table_final_returns.csv", index=False)
display(perf_df.round(3))


,environment,algorithm,n_seeds,mean_final_return,std_final_return,median_final_return,iqm_final_return,ci95_low,ci95_high
0,HalfCheetah-v5,DDPG,5,949.818,477.382,864.609,975.153,534.242,1317.950
1,HalfCheetah-v5,PPO,13,795.697,626.888,1207.535,888.397,461.417,1110.071
2,HalfCheetah-v5,SAC,5,7.525,65.041,-22.162,3.360,-47.696,62.746
3,HalfCheetah-v5,TD3,5,580.466,795.120,24.278,468.175,-83.357,1247.982
4,HalfCheetah-v5,TQC,5,467.173,282.727,596.199,537.680,204.165,671.378
5,Hopper-v5,DDPG,5,213.213,117.306,168.822,164.991,137.009,330.472
6,Hopper-v5,PPO,6,250.489,56.250,229.431,229.125,219.543,301.369
7,Hopper-v5,SAC,5,393.460,173.468,312.024,340.595,268.236,554.572
8,Hopper-v5,TD3,5,136.071,106.386,163.720,124.702,49.591,222.551
9,Hopper-v5,TQC,5,322.181,31.690,336.636,325.555,294.965,349.396


## Sample efficiency and stability


In [5]:
checkpoints = [100_000, 250_000, 500_000]
eff_rows, stab_rows = [], []
for (env_id, algo), part in eval_log.groupby(["environment", "algorithm"]):
    eff = {"environment": env_id, "algorithm": algo}
    for cp in checkpoints:
        closest = min(part["eval_timestep"].unique(), key=lambda x: abs(x - cp))
        seed_means = part[part["eval_timestep"] == closest].groupby("seed")["eval_return"].mean()
        eff[f"mean_return_{cp//1000}k"] = seed_means.mean()
        eff[f"std_return_{cp//1000}k"] = seed_means.std()
    eff_rows.append(eff)

for (env_id, algo), part in final_eval.groupby(["environment", "algorithm"]):
    returns = part["mean_return"].values
    stab_rows.append({"environment": env_id, "algorithm": algo, "std": np.std(returns),
                      "iqr": np.percentile(returns, 75) - np.percentile(returns, 25),
                      "cv": np.std(returns) / (abs(np.mean(returns)) + 1e-8),
                      "failure_rate": part.get("termination_rate", pd.Series([0])).mean()})
eff_df = pd.DataFrame(eff_rows)
stab_df = pd.DataFrame(stab_rows)
if WRITE_TABLES:
    eff_df.to_csv(FINAL_DIR / "table_sample_efficiency.csv", index=False)
if WRITE_TABLES:
    stab_df.to_csv(FINAL_DIR / "table_stability.csv", index=False)
display(eff_df.round(2))
display(stab_df.round(3))


,environment,algorithm,mean_return_100k,std_return_100k,mean_return_250k,std_return_250k,mean_return_500k,std_return_500k
0,HalfCheetah-v5,DDPG,-168.60,405.17,937.54,523.11,937.54,523.11
1,HalfCheetah-v5,PPO,-0.52,0.33,-1.95,8.34,-1.95,8.34
2,HalfCheetah-v5,SAC,-40.98,8.67,1.73,82.53,1.73,82.53
3,HalfCheetah-v5,TD3,-300.32,281.96,572.28,838.41,572.28,838.41
4,HalfCheetah-v5,TQC,-40.80,7.97,475.26,306.91,475.26,306.91
5,Hopper-v5,DDPG,113.97,61.92,221.95,144.99,221.95,144.99
6,Hopper-v5,PPO,118.96,48.34,223.40,11.68,223.40,11.68
7,Hopper-v5,SAC,195.11,25.03,406.91,205.78,406.91,205.78
8,Hopper-v5,TD3,142.54,143.02,139.14,124.03,139.14,124.03
9,Hopper-v5,TQC,204.34,22.32,318.57,33.94,318.57,33.94


,environment,algorithm,std,iqr,cv,failure_rate
0,HalfCheetah-v5,DDPG,477.382,400.254,0.503,0.00
1,HalfCheetah-v5,PPO,626.888,1298.100,0.788,0.00
2,HalfCheetah-v5,SAC,65.041,106.646,8.643,0.00
3,HalfCheetah-v5,TD3,795.120,1529.893,1.370,0.00
4,HalfCheetah-v5,TQC,282.727,254.139,0.605,0.00
5,Hopper-v5,DDPG,117.306,67.851,0.550,1.00
6,Hopper-v5,PPO,56.250,15.282,0.225,1.00
7,Hopper-v5,SAC,173.468,139.092,0.441,1.00
8,Hopper-v5,TD3,106.386,134.992,0.782,1.00
9,Hopper-v5,TQC,31.690,56.972,0.098,1.00


## Probability of improvement and ranking


In [6]:
for env_id, env_data in final_eval.groupby("environment"):
    algos = [a for a in ALGORITHMS if a in env_data["algorithm"].values]
    matrix = pd.DataFrame(index=algos, columns=algos, dtype=float)
    for a in algos:
        for b in algos:
            scores_a = env_data[env_data["algorithm"] == a]["mean_return"].values
            scores_b = env_data[env_data["algorithm"] == b]["mean_return"].values
            matrix.loc[a, b] = 0.5 if a == b else probability_of_improvement(scores_a, scores_b)
    if WRITE_TABLES:
        matrix.to_csv(FINAL_DIR / f"poi_matrix_{env_id.replace('-', '_')}.csv")

rank_df = perf_df.copy()
rank_df["rank_by_iqm"] = rank_df.groupby("environment")["iqm_final_return"].rank(ascending=False)
overall = rank_df.groupby("algorithm")["rank_by_iqm"].mean().reset_index().sort_values("rank_by_iqm")
if WRITE_TABLES:
    overall.to_csv(FINAL_DIR / "table_overall_ranking.csv", index=False)
display(overall.round(3))


,algorithm,rank_by_iqm
2,SAC,2.333
4,TQC,2.333
1,PPO,2.667
0,DDPG,3.333
3,TD3,4.333


## Robustness and AUC


In [7]:
robust_rows = []
for (env_id, algo), part in robust_eval.groupby(["environment", "algorithm"]):
    clean = part[part["noise_sigma"] == 0.0]["eval_return"].mean()
    noisy = part[part["noise_sigma"] == 0.2]["eval_return"].mean()
    robust_rows.append({"environment": env_id, "algorithm": algo,
                        "return_clean": clean, "return_noisy_020": noisy,
                        "drop_pct": (clean - noisy) / (abs(clean) + 1e-8) * 100})
robust_df = pd.DataFrame(robust_rows)
if WRITE_TABLES:
    robust_df.to_csv(FINAL_DIR / "table_robustness.csv", index=False)

auc_rows = []
for (env_id, algo, seed), part in eval_log.groupby(["environment", "algorithm", "seed"]):
    curve = part.groupby("eval_timestep")["eval_return"].mean().sort_index()
    if len(curve) >= 2:
        auc_rows.append({"environment": env_id, "algorithm": algo,
                         "seed": seed, "auc": np.trapz(curve.values, curve.index)})
auc_df = pd.DataFrame(auc_rows)
auc_summary = auc_df.groupby(["environment", "algorithm"])["auc"].agg(["mean", "std"]).reset_index()
if WRITE_TABLES:
    auc_summary.to_csv(FINAL_DIR / "table_auc.csv", index=False)
display(robust_df.round(2))
display(auc_summary.round(2))


C:\Users\digilians01\AppData\Local\Temp\ipykernel_41016\4157233882.py:17: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  "seed": seed, "auc": np.trapz(curve.values, curve.index)})


,environment,algorithm,return_clean,return_noisy_020,drop_pct
0,HalfCheetah-v5,DDPG,853.21,886.64,-3.92
1,HalfCheetah-v5,PPO,765.61,644.92,15.76
2,HalfCheetah-v5,SAC,15.74,-36.41,331.35
3,HalfCheetah-v5,TD3,564.98,394.78,30.12
4,HalfCheetah-v5,TQC,439.67,332.14,24.46
5,Hopper-v5,DDPG,212.36,213.67,-0.61
6,Hopper-v5,PPO,250.71,252.11,-0.56
7,Hopper-v5,SAC,393.95,392.87,0.27
8,Hopper-v5,TD3,131.44,136.80,-4.07
9,Hopper-v5,TQC,322.94,317.83,1.58


,environment,algorithm,mean,std
0,HalfCheetah-v5,DDPG,21381030.87,56737768.05
1,HalfCheetah-v5,PPO,-250936.55,629657.82
2,HalfCheetah-v5,SAC,-2179793.28,2746498.10
3,HalfCheetah-v5,TD3,-27652690.76,63967907.61
4,HalfCheetah-v5,TQC,18942281.02,10057277.82
5,Hopper-v5,DDPG,35120673.67,21575380.52
6,Hopper-v5,PPO,45347161.30,4916614.75
7,Hopper-v5,SAC,58385419.58,7164359.92
8,Hopper-v5,TD3,27596743.83,25061309.60
9,Hopper-v5,TQC,57348459.27,2729610.59
